In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import glob
from pathlib import Path
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

I0000 00:00:1779733141.993879   36084 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


SETUP

In [2]:
PROJECT_ROOT = "/home/hasan/coding/MoneyLens/ai"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
 
from src.ocr_config import *
from src.ocr_model import build_ocr_model, CTCLayer, build_inference_model
from src.text_encoder import (
    encode_text,
    decode_prediction,
    char_to_num,
    num_to_char
)

I0000 00:00:1779733154.023755   36084 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1765 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


KONFIGURASI

In [3]:
BASE_DIR       = os.path.join(PROJECT_ROOT, "Dataset_ocr")
PREP_DIR       = os.path.join(BASE_DIR, "preprocessed")
GT_CSV         = os.path.join(PREP_DIR, "ground_truth_auto.csv")
MODEL_DIR      = os.path.join(PROJECT_ROOT, "saved_model")
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoint")
LOG_DIR        = os.path.join(PROJECT_ROOT, "tensorboard_logs")
 
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
 
BATCH_SIZE    = 32
EPOCHS        = 100
LEARNING_RATE = 1e-4
PATIENCE      = 15

LOAD DATASET

In [4]:
def load_dataset(split: str):
    arrays_dir = os.path.join(PREP_DIR, split, "arrays")
 
    if not os.path.exists(arrays_dir):
        print(f"  [ERROR] Folder tidak ada: {arrays_dir}")
        return None, None, None
 
    if not os.path.exists(GT_CSV):
        print(f"  [ERROR] ground_truth_auto.csv tidak ada: {GT_CSV}")
        return None, None, None
 
    df_gt = pd.read_csv(GT_CSV)
    df_gt = df_gt[df_gt["split"] == split].copy()
    df_gt = df_gt[df_gt["text"].notna() & (df_gt["text"].str.strip() != "")]
 
    fname_to_text = dict(zip(df_gt["filename"], df_gt["text"]))
    npy_count = len(glob.glob(os.path.join(arrays_dir, "*.npy")))
    print(f"  {split}: {len(df_gt)} ground truth | {npy_count} .npy")
 
    images, labels, classes = [], [], []
 
    for npy_path in sorted(glob.glob(os.path.join(arrays_dir, "*.npy"))):
        fname = Path(npy_path).name
        text  = fname_to_text.get(fname, None)
 
        if text is None or str(text).strip() == "":
            continue
 
        try:
            arr = np.load(npy_path, allow_pickle=False)
            if arr.shape != (IMG_H, IMG_W, CHANNELS):
                continue
 
            lbl = encode_text(str(text).lower()).numpy()
 
            if len(lbl) < MAX_TEXT_LENGTH:
                lbl = np.concatenate([
                    lbl,
                    np.full(
                        MAX_TEXT_LENGTH - len(lbl),
                        fill_value=PADDING_VALUE,
                        dtype=np.int32,
                    ),
                ])
            else:
                lbl = lbl[:MAX_TEXT_LENGTH]
 
            images.append(arr)
            labels.append(lbl.astype(np.int32))
            classes.append(fname)
 
        except Exception as e:
            print(f"  [SKIP] {fname}: {e}")
            continue
 
    if not images:
        print(f"  [WARNING] Tidak ada data untuk {split}")
        return None, None, None
 
    print(f"  Loaded: {len(images)} sampel")
    return (
        np.array(images, dtype=np.float32),
        np.array(labels, dtype=np.int32),
        classes,
    )

CUSTOM LOSS

In [ ]:
def ctc_loss_fn(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], tf.int64)
 
    input_len = tf.cast(tf.shape(y_pred)[1], tf.int64) * \
                tf.ones(shape=(batch_len,), dtype=tf.int64)
 
    label_len = tf.reduce_sum(
        tf.cast(tf.not_equal(y_true, PADDING_VALUE), tf.int64), axis=1
    )
 
    return tf.nn.ctc_loss(
        labels=tf.cast(y_true, tf.int32),
        logits=y_pred,
        label_length=tf.cast(label_len, tf.int32),
        logit_length=tf.cast(input_len, tf.int32),
        logits_time_major=False,
        blank_index=BLANK_INDEX
    )

TRAINING LOOP


In [6]:
def train_with_gradient_tape(model, X_train, y_train, X_valid, y_valid, optimizer):
    train_ds = tf.data.Dataset.from_tensor_slices(
        ({"image": X_train, "label": y_train}, y_train)
    ).shuffle(5000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
 
    valid_ds = tf.data.Dataset.from_tensor_slices(
        ({"image": X_valid, "label": y_valid}, y_valid)
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
 
    writer    = tf.summary.create_file_writer(LOG_DIR)
 
    best_loss = np.inf
    wait      = 0
    history   = []
 
    current_lr  = LEARNING_RATE
    lr_patience = 5
    lr_wait     = 0
    lr_factor   = 0.5
    min_lr      = 1e-6
 
    print(f"\n[TRAINING] tf.GradientTape dimulai...")
    print(f"  Epochs     : {EPOCHS}")
    print(f"  Batch size : {BATCH_SIZE}")
    print(f"  Train      : {len(X_train)} sampel")
    print(f"  Valid      : {len(X_valid)} sampel")
    print(f"  LR awal    : {LEARNING_RATE}")
    print(f"  Patience   : {PATIENCE}\n")
 
    for epoch in range(EPOCHS):
 
        # ── Training ──────────────────────────────────────
        train_losses = []
        for batch_x, batch_y in train_ds:
            with tf.GradientTape() as tape:
                y_pred = model(batch_x, training=True)
                loss   = tf.reduce_mean(ctc_loss_fn(
                    tf.cast(batch_x["label"], tf.int32), y_pred
                ))
            grads = tape.gradient(loss, model.trainable_variables)
            # Clip gradients untuk stabilitas
            grads, _ = tf.clip_by_global_norm(grads, 5.0)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            train_losses.append(float(loss))
 
        # ── Validation ────────────────────────────────────
        val_losses = []
        for batch_x, batch_y in valid_ds:
            y_pred = model(batch_x, training=False)
            v_loss = tf.reduce_mean(ctc_loss_fn(
                tf.cast(batch_x["label"], tf.int32), y_pred
            ))
            val_losses.append(float(v_loss))
 
        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
 
        history.append({
            "epoch"    : epoch + 1,
            "loss"     : avg_train,
            "val_loss" : avg_val,
            "lr"       : current_lr,
        })
 
        with writer.as_default():
            tf.summary.scalar("loss",      avg_train,  step=epoch)
            tf.summary.scalar("val_loss",  avg_val,    step=epoch)
            tf.summary.scalar("lr",        current_lr, step=epoch)
 
        print(
            f"  Epoch {epoch+1:03d}/{EPOCHS} "
            f"| loss={avg_train:.4f} "
            f"| val_loss={avg_val:.4f} "
            f"| lr={current_lr:.2e}"
        )

        # ── ReduceLROnPlateau ──────────────────────────────
        if avg_val < best_loss:
            best_loss = avg_val
            wait      = 0
            lr_wait   = 0
            best_path = os.path.join(CHECKPOINT_DIR, "best_model.keras")
            model.save(best_path)
            print(f"    ✅ best_model.keras disimpan (val_loss={avg_val:.4f})")
        else:
            wait    += 1
            lr_wait += 1
 
            # Reduce LR jika tidak ada improvement
            if lr_wait >= lr_patience and current_lr > min_lr:
                current_lr = max(current_lr * lr_factor, min_lr)
                optimizer.learning_rate.assign(current_lr)
                lr_wait = 0
                print(f"    📉 LR diturunkan → {current_lr:.2e}")
 
            if wait >= PATIENCE:
                print(f"    🛑 Early stopping di epoch {epoch+1}")
                break
 
    return history

MAIN

In [7]:
if __name__ == "__main__":
    print("[DATA] Memuat dataset dari ground_truth_auto.csv...")
    X_train, y_train, cls_train = load_dataset("train")
    X_valid, y_valid, cls_valid = load_dataset("valid")
    X_test,  y_test,  cls_test  = load_dataset("test")
 
    print()
    for split, X in [("train", X_train), ("valid", X_valid), ("test", X_test)]:
        if X is not None:
            print(f"  {split:5s}: {len(X):4d} sampel → shape={X.shape}")
        else:
            print(f"  {split:5s}: tidak ada data ❌")
 
    print(f"\n[MODEL] Membangun model...")
    model = build_ocr_model()
    print(f"  Params: {model.count_params():,}")
    model.summary()
 
    if X_train is not None and X_valid is not None:
        optimizer = keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=5.0
        )
 
        history = train_with_gradient_tape(
            model, X_train, y_train,
            X_valid, y_valid,
            optimizer=optimizer
        )
 
        pd.DataFrame(history).to_csv(
            os.path.join(PROJECT_ROOT, "training_history.csv"),
            index=False
        )
 
        config = {
            "model_name"   : "MoneyLens_OCR",
            "architecture" : "CNN(32-64-128) + BiLSTM(128-64) + CTC",
            "num_classes"  : NUM_CLASSES,
            "blank_index"  : BLANK_INDEX,
            "padding_value": PADDING_VALUE,
            "training": {
                "epochs_trained": len(history),
                "best_val_loss" : float(min(h["val_loss"] for h in history)),
                "batch_size"    : BATCH_SIZE,
                "learning_rate" : LEARNING_RATE,
                "bugs_fixed"    : [
                    "blank_index=0 konsisten di seluruh pipeline",
                    "padding_value=-1 (bukan NUM_CLASSES)",
                    "compute_metrics index mapping diperbaiki",
                    "softmax sebelum ctc_decode",
                    "build_inference_model robust via get_layer()",
                    "LR 1e-3 -> 1e-4 + ReduceLROnPlateau",
                ]
            },
            "data": {
                "train_samples": len(X_train),
                "valid_samples": len(X_valid),
                "test_samples" : len(X_test) if X_test is not None else 0,
                "ground_truth" : GT_CSV,
            }
        }
        with open(os.path.join(MODEL_DIR, "training_config.json"), "w") as f:
            json.dump(config, f, indent=2)
 
    else:
        print("\n[ERROR] Data tidak tersedia!")
        print(f"  Pastikan ground_truth_auto.csv ada di: {GT_CSV}")
 
    print(f"\n{'='*65}\nTRAINING SELESAI\n{'='*65}")
    print(f"  Checkpoint  : {CHECKPOINT_DIR}/best_model.keras")
    print(f"  TensorBoard : tensorboard --logdir={LOG_DIR}")
    print(f"  History     : {os.path.join(PROJECT_ROOT, 'training_history.csv')}")
    print(f"{'='*65}")

[DATA] Memuat dataset dari ground_truth_auto.csv...
  train: 3190 ground truth | 3194 .npy
  Loaded: 3172 sampel
  valid: 915 ground truth | 915 .npy
  Loaded: 915 sampel
  test: 422 ground truth | 422 .npy
  Loaded: 422 sampel

  train: 3172 sampel → shape=(3172, 32, 128, 1)
  valid:  915 sampel → shape=(915, 32, 128, 1)
  test :  422 sampel → shape=(422, 32, 128, 1)

[MODEL] Membangun model...
  Params: 499,478


Model: "MoneyLens_OCR"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 32, 128,   │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 128,   │        320 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 128,   │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16, 64,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 64,    │     18,496 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 8, 32, 64) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 8, 32, 64) │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 8, 32,     │     73,856 │ dropout_1[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 32,     │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 4, 32,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 4, 32,     │          0 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 32, 512)   │          0 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32, 64)    │     32,832 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32, 64)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 32, 256)   │    197,632 │ dropout_3[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 32, 128)   │    164,352 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │                 

 Total params: 499,478 (1.91 MB)

 Trainable params: 499,030 (1.90 MB)

 Non-trainable params: 448 (1.75 KB)


[TRAINING] tf.GradientTape dimulai...
  Epochs     : 100
  Batch size : 32
  Train      : 3172 sampel
  Valid      : 915 sampel
  LR awal    : 0.0001
  Patience   : 15



I0000 00:00:1779733223.937111   36084 cuda_dnn.cc:461] Loaded cuDNN version 92200


  Epoch 001/100 | loss=69.4083 | val_loss=56.2188 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=56.2188)
  Epoch 002/100 | loss=36.8459 | val_loss=54.5011 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=54.5011)
  Epoch 003/100 | loss=34.2362 | val_loss=54.0717 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=54.0717)
  Epoch 004/100 | loss=33.3930 | val_loss=51.8206 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=51.8206)
  Epoch 005/100 | loss=32.9258 | val_loss=51.1659 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=51.1659)
  Epoch 006/100 | loss=32.6123 | val_loss=49.0275 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=49.0275)
  Epoch 007/100 | loss=32.1886 | val_loss=48.7850 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=48.7850)
  Epoch 008/100 | loss=31.8673 | val_loss=48.7398 | lr=1.00e-04
    ✅ best_model.keras disimpan (val_loss=48.7398)
  Epoch 009/100 | loss=32.0642 | val_loss=48.1358 | lr=1.00e-04
    ✅ best_model

In [10]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import sys
sys.path.insert(0, "/home/hasan/coding/MoneyLens/ai")

tf.keras.mixed_precision.set_global_policy("float32")

from src.ocr_config import *
from src.ocr_model import build_ocr_model, build_inference_model
from src.text_encoder import char_to_num

# ✅ Rebuild arsitektur baru, lalu load weights saja
model = build_ocr_model()
model.load_weights(
    "/home/hasan/coding/MoneyLens/ai/saved_model/checkpoint/best_model.keras"
)
inf_model = build_inference_model(model)

# Vocab untuk decode
vocab       = char_to_num.get_vocabulary()
idx_to_char = {i: c for i, c in enumerate(vocab)}

# Load sampel valid
import os
import pandas as pd

PREP_DIR = "/home/hasan/coding/MoneyLens/ai/Dataset_ocr/preprocessed"
df       = pd.read_csv(os.path.join(PREP_DIR, "ground_truth_auto.csv"))
df_val   = df[df["split"] == "valid"].dropna(subset=["text"]).head(10)

print(f"{'GT':<30} {'PRED':<30} {'MATCH'}")
print("-" * 70)

for _, row in df_val.iterrows():
    npy_path = os.path.join(PREP_DIR, "valid", "arrays", row["filename"])
    if not os.path.exists(npy_path):
        continue

    arr       = np.load(npy_path)[np.newaxis].astype(np.float32)
    logits    = inf_model.predict(arr, verbose=0)
    probs     = tf.nn.softmax(logits, axis=-1).numpy()
    input_len = np.array([probs.shape[1]], dtype=np.int32)

    decoded, _ = keras.backend.ctc_decode(
        probs, input_length=input_len, greedy=True
    )
    indices = decoded[0].numpy()[0]

    pred  = "".join([idx_to_char.get(int(i), "") for i in indices if i > 0]).strip()
    gt    = str(row["text"]).strip()
    match = "✅" if pred == gt else "❌"
    print(f"{gt:<30} {pred:<30} {match}")

GT                             PRED                           MATCH
----------------------------------------------------------------------
8,20                                                          ❌
24MMx7Y M.ONE TAPE                                            ❌
313,50                                                        ❌
114,00                                                        ❌
50,00                                                         ❌
1                              1                              ✅
PC 0/0 H&H                                                    ❌
15                             1                              ❌
5 30                                                          ❌
1                                                             ❌


In [11]:
import tensorflow as tf
print(tf.keras.mixed_precision.global_policy())

<DTypePolicy "float32">


In [12]:
import tensorflow as tf
import numpy as np
import sys
sys.path.insert(0, "/home/hasan/coding/MoneyLens/ai")

# 1. Cek mixed precision
policy = tf.keras.mixed_precision.global_policy()
print(f"[1] Mixed precision : {policy.name}")
print(f"    {'✅ OK (float32)' if policy.name == 'float32' else '❌ HARUS DIMATIKAN — jalankan: tf.keras.mixed_precision.set_global_policy(float32)'}")

# 2. Cek GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\n[2] GPU tersedia    : {len(gpus)}")
for g in gpus:
    print(f"    {g}")

# 3. Cek blank index dan vocab konsisten
from src.ocr_config import BLANK_INDEX, NUM_CLASSES, PADDING_VALUE, CHARACTERS
from src.text_encoder import char_to_num
vocab = char_to_num.get_vocabulary()
print(f"\n[3] BLANK_INDEX     : {BLANK_INDEX}")
print(f"    NUM_CLASSES     : {NUM_CLASSES}")
print(f"    PADDING_VALUE   : {PADDING_VALUE}")
print(f"    Vocab size      : {len(vocab)}")
print(f"    vocab[0]        : '{vocab[0]}' (harus [UNK]/blank)")
print(f"    vocab[1]        : '{vocab[1]}' (harus karakter pertama)")
print(f"    {'✅ OK' if vocab[0] == '[UNK]' and len(vocab) == NUM_CLASSES else '❌ Ada masalah vocab'}")

# 4. Cek encode/decode round-trip
from src.text_encoder import encode_text
test_str = "13.500,00"
encoded  = encode_text(test_str).numpy()
idx_to_char = {i: c for i, c in enumerate(vocab)}
decoded  = "".join([idx_to_char.get(int(i), "?") for i in encoded if i > 0])
print(f"\n[4] Encode-decode test")
print(f"    Input   : '{test_str}'")
print(f"    Encoded : {encoded}")
print(f"    Decoded : '{decoded}'")
print(f"    {'✅ OK' if decoded == test_str else '❌ Encode/decode tidak konsisten'}")

[1] Mixed precision : float32
    ✅ OK (float32)

[2] GPU tersedia    : 1
    PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

[3] BLANK_INDEX     : 0
    NUM_CLASSES     : 86
    PADDING_VALUE   : -1
    Vocab size      : 86
    vocab[0]        : '[UNK]' (harus [UNK]/blank)
    vocab[1]        : '0' (harus karakter pertama)
    ✅ OK

[4] Encode-decode test
    Input   : '13.500,00'
    Encoded : [ 2  4 64  6  1  1 65  1  1]
    Decoded : '13.500,00'
    ✅ OK
